# Medallion Architecture — Delta Live Tables Pipeline

This notebook defines the **Bronze → Silver → Gold** pipeline declaratively
using Delta Live Tables (DLT). It is uploaded to the Databricks workspace
and referenced as the source code in the `medallion_dlt_pipeline` DLT pipeline.

**Pipeline name:** `medallion_dlt_pipeline`
**Pipeline mode:** Triggered
**Destination schema:** `phase8_lab`

> Before creating the pipeline, ensure the path variables below match
> the S3 bucket you created in Activity 1.

## Configuration

Update `bucket` to your actual S3 bucket name before running the pipeline.

In [0]:
import dlt
from pyspark.sql.functions import (
    col,
    current_timestamp,
    upper,
    trim,
    to_timestamp,
    to_date,
    sum as _sum,
    count,
    countDistinct,
    row_number,
)
from pyspark.sql.window import Window

# ---------------------------------------------------------------------------
# Paths — replace <your-initials>-<random-4-digits> with your actual suffix
# ---------------------------------------------------------------------------
bucket            = "phase8-databricks-lakehouse"
raw_orders_path   = f"s3://{bucket}/raw/orders/"
bronze_checkpoint = f"s3://{bucket}/checkpoints/bronze_orders/"

## Bronze Layer

Incrementally ingests CSV files from the S3 raw landing path using Auto Loader.
Adds `source_file` and `ingestion_ts` metadata columns so every record is
traceable back to its origin file and load time.

In [0]:
@dlt.table(
    name="dlt_bronze_orders",
    comment="Bronze ingestion from S3 raw/orders/ using Auto Loader (cloudFiles). "
            "Preserves raw fidelity and captures ingestion metadata.",
)
def dlt_bronze_orders():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")
             .option("cloudFiles.schemaLocation", bronze_checkpoint + "_dlt_schema")
             .option("header", "true")
             .load(raw_orders_path)
             .withColumn("source_file", col("_metadata.file_path"))
             .withColumn("ingestion_ts", current_timestamp())
    )

## Silver Layer

Cleanses and deduplicates Bronze records.

**Data Quality Expectations (enforce via `expect_or_drop`):**
| Rule | Column | Condition |
|------|--------|-----------|
| valid_order_id | order_id | IS NOT NULL |
| valid_customer_id | customer_id | IS NOT NULL |
| valid_status | order_status | IN ('PLACED','SHIPPED','DELIVERED','CANCELLED') |

Rows that fail any expectation are **dropped** before reaching the Silver table.
A window function retains only the **latest** record per `order_id`.

In [0]:
@dlt.table(
    name="dlt_silver_orders",
    comment="Silver cleansed and deduplicated orders. "
            "Invalid rows dropped by DLT expectations. "
            "One row per order_id (latest by order_ts + ingestion_ts).",
)
@dlt.expect_or_drop("valid_order_id",    "order_id IS NOT NULL AND order_id != ''")
@dlt.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL AND customer_id != ''")
@dlt.expect_or_drop(
    "valid_status",
    "upper(trim(status)) IN ('PLACED','SHIPPED','DELIVERED','CANCELLED')",
)
def dlt_silver_orders():
    df = (
        dlt.read("dlt_bronze_orders")
           .withColumn("order_status", upper(trim(col("status"))))
           .withColumn("order_ts",     to_timestamp(col("order_ts")))
    )

    dedup_window = (
        Window.partitionBy("order_id")
              .orderBy(col("order_ts").desc(), col("ingestion_ts").desc())
    )

    return (
        df.withColumn("rn", row_number().over(dedup_window))
          .filter(col("rn") == 1)
          .drop("rn")
    )

## Gold Layer

Aggregates Silver records into **daily business metrics** grouped by
`order_date` and `order_status`. Produces three KPI columns:
- `total_orders`       — count of orders
- `distinct_customers` — count of unique customers

In [0]:
@dlt.table(
    name="dlt_gold_daily_sales",
    comment="Gold business metrics aggregated by order_date and order_status. "
            "Ready for Databricks SQL dashboards and reporting.",
)
def dlt_gold_daily_sales():
    return (
        dlt.read("dlt_silver_orders")
           .withColumn("order_date", to_date(col("order_ts")))
           .groupBy("order_date", "status")
           .agg(
               count("order_id").alias("total_orders"),
               countDistinct("customer_id").alias("distinct_customers"),
           )
    )